# Phase 37 — Expert-label provenance and external dataset audit
This notebook retrieves the **QYTL human-annotation files**, inspects actual columns, label distributions and overlap, and produces a ZIP of evidence. **No Bloom taxonomy mapping is assumed.** It does not report external model accuracy without an explicit validated mapping and predictions. Run on Google Colab (CPU sufficient).

In [ ]:
!pip -q install pandas requests

In [ ]:
from pathlib import Path
import requests, pandas as pd, json, hashlib
root=Path('/content/phase37'); root.mkdir(exist_ok=True)
base='https://raw.githubusercontent.com/YesNLP/QYTL/main/data/q_bt_human/'
for name in ['q_bt_human_label.csv','q_bt_human_label_raw.csv']:
    response=requests.get(base+name,timeout=90); response.raise_for_status()
    (root/name).write_bytes(response.content)
    print(name,len(response.content),'bytes',hashlib.sha256(response.content).hexdigest())

In [ ]:
report={}
for f in sorted(root.glob('*.csv')):
    df=pd.read_csv(f)
    print('\n',f.name,'shape',df.shape,'columns',list(df.columns))
    print(df.head(3).to_string(index=False,max_colwidth=55))
    info={'rows':len(df),'columns':list(df.columns),'missing':df.isna().sum().to_dict(), 'sha256':hashlib.sha256(f.read_bytes()).hexdigest()}
    for c in df.columns:
        if df[c].nunique(dropna=True)<=25:
            info.setdefault('low_cardinality_columns',{})[c]={str(k):int(v) for k,v in df[c].value_counts(dropna=False).items()}
    report[f.name]=info
(root/'audit.json').write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str))
print('\nAudit saved')

## Interpretation gate
Before using these labels as an external benchmark, inspect the source paper for annotator expertise, number of annotators, adjudication, and the exact taxonomy. Do not assume human annotations are expert consensus. The earlier DistilBERT ZIP contains only test predictions, not trained weights; therefore this notebook **cannot** infer DistilBERT on these new questions. Retraining/saving a model is required for external inference.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/Phase37_Human_Annotation_Audit','zip',root_dir='/content',base_dir='phase37')
print(archive)
files.download(archive)